# [6.3] Transcoders and Attribution Graphs - Exercises

Implement the local transcoder and attribution-graph primitives. The CUDA report for this section uses pinned TransformerLens `gelu-1l`, but these exercises are CPU-safe and focus on the exact contracts that the real-model path depends on.


In [ ]:
import sys
from dataclasses import dataclass
from pathlib import Path

import torch as t
import torch.nn.functional as F

chapter = "chapter6_sparse_feature_methods"
section = "part3_transcoders_attribution_graphs"
root_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section
if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part3_transcoders_attribution_graphs.tests as tests


@dataclass(frozen=True)
class TranscoderOutput:
    feature_acts: t.Tensor
    reconstructed_activations: t.Tensor


@dataclass(frozen=True)
class TranscoderReplacementReport:
    reconstruction_mse: float
    replacement_kl: float
    target_logit_diff: float
    replacement_logit_diff: float
    logit_diff_error: float
    passes_kl: bool
    preserves_logit_diff: bool


@dataclass(frozen=True)
class AttributionEdge:
    source_type: str
    source_id: int
    target_type: str
    target_id: int
    weight: float


@dataclass(frozen=True)
class AttributionGraphReport:
    num_nodes: int
    num_edges: int
    density: float
    full_logit_diff: float
    graph_logit_diff: float
    ablated_logit_diff: float
    random_ablated_logit_diff: float
    topk_damage: float
    random_damage: float
    preserves_logit_diff: bool
    passes_damage_control: bool
    reproducible: bool


## Transcoder Forward

Implement a ReLU encoder followed by a linear decoder. Keep the feature dimension convention explicit: `encoder_weight` has shape `(features, d_in)` and `decoder_weight` has shape `(features, d_out)`.


In [ ]:
def transcoder_forward(
    inputs: t.Tensor,
    encoder_weight: t.Tensor,
    decoder_weight: t.Tensor,
    *,
    encoder_bias: t.Tensor | None = None,
    decoder_bias: t.Tensor | None = None,
) -> TranscoderOutput:
    raise NotImplementedError()


tests.test_transcoder_forward_matches_reference_and_relu_rules(transcoder_forward)


## Replacement Metrics

A transcoder is useful only if replacing the original component preserves the relevant downstream behavior. Implement KL, target logit difference, and the replacement report.


In [ ]:
def mean_kl_divergence(reference_logits: t.Tensor, reconstructed_logits: t.Tensor) -> float:
    raise NotImplementedError()


def target_logit_diff(
    logits: t.Tensor,
    *,
    positive_token_id: int,
    negative_token_id: int,
) -> float:
    raise NotImplementedError()


def transcoder_replacement_report(
    *,
    reference_activations: t.Tensor,
    reconstructed_activations: t.Tensor,
    reference_logits: t.Tensor,
    replacement_logits: t.Tensor,
    positive_token_id: int,
    negative_token_id: int,
    kl_threshold: float = 1e-3,
    logit_diff_tolerance: float = 0.1,
) -> TranscoderReplacementReport:
    raise NotImplementedError()


tests.test_target_logit_diff_and_replacement_report_match_reference(
    target_logit_diff,
    transcoder_replacement_report,
)


## Feature Contributions

Reduce every non-feature dimension, then multiply by direct logit effects. This gives a graph hypothesis, not causal proof.


In [ ]:
def feature_logit_contributions(feature_acts: t.Tensor, logit_effects: t.Tensor) -> t.Tensor:
    raise NotImplementedError()


tests.test_feature_logit_contributions_reduce_all_nonfeature_dimensions(
    feature_logit_contributions,
)


## Graph Edges

Build a deterministic edge list with signed weights. Select by absolute magnitude so large negative effects are not dropped.


In [ ]:
def build_attribution_edges(
    input_to_feature_scores: t.Tensor,
    feature_logit_effects: t.Tensor,
    *,
    top_k: int,
) -> list[AttributionEdge]:
    raise NotImplementedError()


def graph_reproducible(
    edges_a: list[AttributionEdge],
    edges_b: list[AttributionEdge],
    *,
    atol: float = 1e-6,
) -> bool:
    raise NotImplementedError()


tests.test_build_attribution_edges_keeps_top_input_and_logit_edges(
    build_attribution_edges,
    graph_reproducible,
)
tests.test_graph_reproducible_rejects_structure_and_weight_changes(
    AttributionEdge,
    graph_reproducible,
)


## Graph Reports

Check whether selected graph features preserve most of the target logit difference and whether removing them damages behavior more than removing low-effect features.


In [ ]:
def graph_density(*, num_nodes: int, num_edges: int) -> float:
    raise NotImplementedError()


def attribution_graph_report(
    contributions: t.Tensor,
    graph_feature_ids: t.Tensor | list[int],
    random_feature_ids: t.Tensor | list[int],
    *,
    num_nodes: int,
    num_edges: int,
    reproducible: bool,
    preservation_threshold: float = 0.8,
) -> AttributionGraphReport:
    raise NotImplementedError()


tests.test_attribution_graph_report_preservation_and_damage_controls(
    graph_density,
    attribution_graph_report,
)


## Full Verification

After filling in the notebook, compare your implementation to `solutions.py`. The full CUDA path is run separately by `solutions.run_gpu_test(max_vram_gb=24.0)` and should report oracle replacement parity, trained transcoder reconstruction, graph preservation, graph damage controls, and peak VRAM.


## Full Verification Contract

The smoke tests check the local exercise implementation. This final cell checks the committed CUDA verification report for the section-scale run and exposes the same `run_gpu_test` / `run_full_experiment` surface used by the release gate.


In [ ]:
def _load_committed_gpu_report() -> dict:
    import json

    report = json.loads((section_dir / "verification_report.json").read_text())
    assert report["accepted"] and report["tests_passed"]
    gpu = report["metrics"]["gpu_test"]
    assert gpu["cuda_available"]
    assert gpu["within_vram_budget"]
    return gpu


def run_gpu_test(max_vram_gb: float = 24.0) -> dict:
    gpu = _load_committed_gpu_report()
    assert gpu["peak_vram_gb"] <= max_vram_gb
    return gpu


def run_full_experiment(max_vram_gb: float = 24.0) -> dict:
    return run_gpu_test(max_vram_gb=max_vram_gb)


gpu = _load_committed_gpu_report()
{key: gpu[key] for key in [
    "device",
    "preflight_passed",
    "peak_vram_gb",
] if key in gpu}
